In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as transforms
from torchvision import datasets
import torch.optim as optim

# defining the transformer 

In [2]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

In [3]:
#loading the dataset 

dataset = datasets.ImageFolder(
    root="data",
    transform=transform
)

In [4]:
dataset

Dataset ImageFolder
    Number of datapoints: 24998
    Root location: data
    StandardTransform
Transform: Compose(
               Resize(size=(128, 128), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
print(dataset.classes)
print(dataset.class_to_idx)
print(len(dataset))

['Cat', 'Dog']
{'Cat': 0, 'Dog': 1}
24998


In [6]:
image, label = dataset[0]

print(image.shape)
print(label)

torch.Size([3, 128, 128])
0


# spliting the data into train and test set 

In [7]:
train_size = int(0.8 *len(dataset))
test_size = len(dataset)- train_size

In [8]:
train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

In [9]:
train_loader = DataLoader(dataset= train_dataset, batch_size= 32,  shuffle = True)
test_loader = DataLoader(dataset= test_dataset, batch_size= 32)

# model building

In [12]:
class catDogCNN(nn.Module):
    def __init__(self):
        super(catDogCNN, self).__init__()

        self.conv_layers = nn.Sequential(

            nn.Conv2d(3,32, 3, 1,padding=1), #in channel 3, out channel is 32, kernel size is 3, stride 1.
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64, 3, 1,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128, 3, 1,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )


        self.fully_connected_layer = nn.Sequential(
            
            nn.Flatten(),
            nn.Linear(16*16*128, 128),
            nn.ReLU(),

            nn.Linear(128, 1)
            
        )


    def forward(self, x):
        x= self.conv_layers(x)
        x= self.fully_connected_layer(x)

        return x
        

In [13]:
model = catDogCNN()

x = torch.randn(32, 3, 128, 128)

print(model.conv_layers(x).shape)

torch.Size([32, 128, 16, 16])


# Train the model

In [19]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters())

In [23]:
epochs = 10

for epoch in range(epochs):

    epoch_train_loss = 0.0

    for img, label in train_loader:
        
        optimizer.zero_grad()

        output = model.forward(img)

        label = label.float().unsqueeze(1)

        loss = criterion(output, label)

        loss.backward()
        optimizer.step()

        epoch_train_loss += loss.item()

    print(f"epoch = {epoch} & loss = {epoch_train_loss}")


epoch = 0 & loss = 282.5003686398268
epoch = 1 & loss = 224.7219738215208
epoch = 2 & loss = 178.21226985007524
epoch = 3 & loss = 126.27993389591575
epoch = 4 & loss = 71.89310041256249
epoch = 5 & loss = 41.335487524280325
epoch = 6 & loss = 25.31773581629386
epoch = 7 & loss = 23.328666205285117
epoch = 8 & loss = 15.02383042567817
epoch = 9 & loss = 22.776090876002854


# model evaluation

In [2]:
def evaluate(model, dataloader):
    
    model.eval()
    correct_count= 0
    total_count = 0

    with torch.no_grad():
        for img, label in dataloader:

            outpurt = model(img)
            predictions = (torch.sigmoid(output)>0.5).float()

            label = label.float().unsqueeze(1)

            correct_count+= (predictions==label).sum().item()

            total_count+= label.size(0)

        accuracy = 100 * correct / total
        return accuracy

    


In [ ]:
test_accuracy = evaluate(model, test_loader)

print(f"Test Accuracy: {test_accuracy:.2f}%")